# Pandas II — Cleaning, GroupBy & Merge

**Objective:** Transform messy data into an analysis-ready dataset.

In [1]:
import pandas as pd
import numpy as np

## 1. Raw messy dataset

In [2]:
raw = pd.DataFrame({
    "order_id": [1001, 1002, 1003, 1004, 1005, 1006, 1007, 1008],
    "date": ["2026-01-03", "2026/01/05", "Jan 07 2026", "2026-01-08",
             None, "2026-01-15", "2026-01-18", "2026-01-20"],
    "region": ["Karachi", "lahore", "KARACHI", "Islamabad", "Lahore", None, "karachi", "Islamabad"],
    "product": ["Laptop", "Phone", " laptop ", "Tablet", "Phone", "Headphones", "Phone", None],
    "quantity": ["2", "3", "1", "2", "four", "5", None, "3"],
    "unit_price": [120000, 65000, "120000", 45000, 65000, 8000, 65000, None],
    "rating": [4.5, 4.0, None, 4.2, 3.9, 4.1, 4.0, 4.3]
})
display(raw)
print("\nMissing values:")
display(raw.isna().sum())

,order_id,date,region,product,quantity,unit_price,rating
0,1001,2026-01-03,Karachi,Laptop,2,120000,4.5
1,1002,2026/01/05,lahore,Phone,3,65000,4.0
2,1003,Jan 07 2026,KARACHI,laptop,1,120000,NaN
3,1004,2026-01-08,Islamabad,Tablet,2,45000,4.2
4,1005,NaN,Lahore,Phone,four,65000,3.9
5,1006,2026-01-15,NaN,Headphones,5,8000,4.1
6,1007,2026-01-18,karachi,Phone,NaN,65000,4.0
7,1008,2026-01-20,Islamabad,NaN,3,None,4.3



Missing values:


order_id      0
date          1
region        1
product       1
quantity      1
unit_price    1
rating        1
dtype: int64

## 2. Clean strings and standardize categories

In [3]:
clean = raw.copy()

clean["region"] = clean["region"].str.strip().str.title()
clean["product"] = clean["product"].str.strip().str.title()

display(clean[["region", "product"]])

,region,product
0,Karachi,Laptop
1,Lahore,Phone
2,Karachi,Laptop
3,Islamabad,Tablet
4,Lahore,Phone
5,NaN,Headphones
6,Karachi,Phone
7,Islamabad,NaN


## 3. Convert types

In [4]:
clean["date"] = pd.to_datetime(clean["date"], errors="coerce")
clean["quantity"] = pd.to_numeric(clean["quantity"], errors="coerce")
clean["unit_price"] = pd.to_numeric(clean["unit_price"], errors="coerce")

clean.dtypes

order_id               int64
date          datetime64[us]
region                   str
product                  str
quantity             float64
unit_price           float64
rating               float64
dtype: object

## 4. Handle missing data

In [5]:
clean["quantity"] = clean["quantity"].fillna(clean["quantity"].median())
clean["unit_price"] = clean["unit_price"].fillna(clean.groupby("product")["unit_price"].transform("median"))
clean["rating"] = clean["rating"].fillna(clean["rating"].median())
clean["region"] = clean["region"].fillna("Unknown")
clean["product"] = clean["product"].fillna("Unknown")
clean = clean.dropna(subset=["date"])

clean["unit_price"] = clean["unit_price"].fillna(clean["unit_price"].median())

display(clean)
print("\nRemaining missing values:")
display(clean.isna().sum())

,order_id,date,region,product,quantity,unit_price,rating
0,1001,2026-01-03,Karachi,Laptop,2.0,120000.0,4.5
3,1004,2026-01-08,Islamabad,Tablet,2.0,45000.0,4.2
5,1006,2026-01-15,Unknown,Headphones,5.0,8000.0,4.1
6,1007,2026-01-18,Karachi,Phone,2.5,65000.0,4.0
7,1008,2026-01-20,Islamabad,Unknown,3.0,55000.0,4.3



Remaining missing values:


order_id      0
date          0
region        0
product       0
quantity      0
unit_price    0
rating        0
dtype: int64

## 5. Feature engineering

In [6]:
clean["revenue"] = clean["quantity"] * clean["unit_price"]
clean["month"] = clean["date"].dt.month
clean["day_name"] = clean["date"].dt.day_name()

display(clean)

,order_id,date,region,product,quantity,unit_price,rating,revenue,month,day_name
0,1001,2026-01-03,Karachi,Laptop,2.0,120000.0,4.5,240000.0,1,Saturday
3,1004,2026-01-08,Islamabad,Tablet,2.0,45000.0,4.2,90000.0,1,Thursday
5,1006,2026-01-15,Unknown,Headphones,5.0,8000.0,4.1,40000.0,1,Thursday
6,1007,2026-01-18,Karachi,Phone,2.5,65000.0,4.0,162500.0,1,Sunday
7,1008,2026-01-20,Islamabad,Unknown,3.0,55000.0,4.3,165000.0,1,Tuesday


## 6. GroupBy aggregation

In [7]:
region_summary = clean.groupby("region").agg(
    orders=("order_id", "count"),
    units=("quantity", "sum"),
    revenue=("revenue", "sum"),
    avg_rating=("rating", "mean")
).reset_index()

display(region_summary)

,region,orders,units,revenue,avg_rating
0,Islamabad,2,5.0,255000.0,4.25
1,Karachi,2,4.5,402500.0,4.25
2,Unknown,1,5.0,40000.0,4.10


## 7. Pivot table

In [8]:
pivot = pd.pivot_table(
    clean,
    values="revenue",
    index="region",
    columns="product",
    aggfunc="sum",
    fill_value=0
)
display(pivot)

product,Headphones,Laptop,Phone,Tablet,Unknown
region,,,,,
Islamabad,0.0,0.0,0.0,90000.0,165000.0
Karachi,0.0,240000.0,162500.0,0.0,0.0
Unknown,40000.0,0.0,0.0,0.0,0.0


## 8. Merge with product metadata

In [9]:
product_info = pd.DataFrame({
    "product": ["Laptop", "Phone", "Tablet", "Headphones", "Unknown"],
    "category": ["Computing", "Mobile", "Computing", "Audio", "Other"]
})

merged = clean.merge(product_info, on="product", how="left")
display(merged)

,order_id,date,region,product,quantity,unit_price,rating,revenue,month,day_name,category
0,1001,2026-01-03,Karachi,Laptop,2.0,120000.0,4.5,240000.0,1,Saturday,Computing
1,1004,2026-01-08,Islamabad,Tablet,2.0,45000.0,4.2,90000.0,1,Thursday,Computing
2,1006,2026-01-15,Unknown,Headphones,5.0,8000.0,4.1,40000.0,1,Thursday,Audio
3,1007,2026-01-18,Karachi,Phone,2.5,65000.0,4.0,162500.0,1,Sunday,Mobile
4,1008,2026-01-20,Islamabad,Unknown,3.0,55000.0,4.3,165000.0,1,Tuesday,Other


## 9. map/apply

In [10]:
rating_label = {
    "Laptop": "Premium",
    "Phone": "Popular",
    "Tablet": "Computing",
    "Headphones": "Audio",
    "Unknown": "Other"
}
merged["product_segment"] = merged["product"].map(rating_label)

merged["revenue_k"] = merged["revenue"].apply(lambda x: round(x / 1000, 1))
display(merged[["product", "revenue", "revenue_k", "product_segment"]])

,product,revenue,revenue_k,product_segment
0,Laptop,240000.0,240.0,Premium
1,Tablet,90000.0,90.0,Computing
2,Headphones,40000.0,40.0,Audio
3,Phone,162500.0,162.5,Popular
4,Unknown,165000.0,165.0,Other


## Conclusion

The raw dataset was standardized, converted to appropriate types, cleaned for missing values, enriched with derived features, aggregated with GroupBy/pivot_table, and joined with metadata using merge.